# ML Pipeline 8: In-Kind Donation Needs Forecasting & Gap Analysis

## Problem Framing

**Business Question**: Are in-kind donations matching the organization's actual needs? Which item categories are over-supplied vs. critically under-supplied?

**Why It Matters**: In-kind donations (food, school materials, medical supplies, hygiene items) are operationally critical but unpredictable. Ember needs to know: what to ask donors for, when seasonal spikes occur, and whether current donations align with the program's real needs (Health, Shelter, Hygiene, Education).

**Modeling Approach**:
- **Descriptive**: Item category distribution â€” what are donors giving vs. what's needed?
- **Seasonal Analysis**: When do in-kind donations spike? What items come in which seasons?
- **Predictive**: Forecast next-month in-kind volume and item mix by category
- **Gap Analysis**: Compare intended use (what donors say items are for) vs. categories received

**Success Metrics**:
- Identify top 2 categories with supply-demand gaps
- Forecast monthly in-kind item volumes with <20% MAPE
- Produce a 'wish list' of most-needed categories for donor communications

In [ ]:
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# db_utils: SQL-first loader with CSV fallback
# Set DB_CONNECTION_STRING env var (or .env file in ml-pipelines/) to use Azure SQL.
# Automatically falls back to the local CSV files if SQL is unavailable.
sys.path.insert(0, str(pathlib.Path().resolve()))
import db_utils as db
print(f'Data source: {db.connection_status()}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV

# Load data (SQL first, CSV fallback)
items, donations, safehouses, safehouse_metrics = db.load_tables(
    'in_kind_donation_items', 'donations',
    'safehouses', 'safehouse_monthly_metrics'
)
print(f'Items:           {items.shape}')
print(f'Safehouses:      {safehouses.shape}')
print(f'Monthly metrics: {safehouse_metrics.shape}')


## 1. Category & Value Distribution

In [ ]:
# Summary by category
category_summary = items.groupby('item_category').agg(
    item_count=('item_id', 'count'),
    total_quantity=('quantity', 'sum'),
    total_value=('estimated_unit_value', lambda x: (x * items.loc[x.index, 'quantity']).sum()),
    avg_unit_value=('estimated_unit_value', 'mean'),
    pct_new=('received_condition', lambda x: (x == 'New').sum() / len(x))
).reset_index().sort_values('total_value', ascending=False)

print("=== IN-KIND CATEGORY SUMMARY ===")
print(category_summary.to_string(index=False))

# Intended use vs. category cross-tab
print("\nCross-tab: Item Category vs. Intended Use:")
cross = pd.crosstab(items['item_category'], items['intended_use'])
print(cross)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('In-Kind Donation Analysis', fontsize=15, fontweight='bold')

colors = ['#E8641A', '#0D4C5E', '#C4A24B', '#6B7280']

# Total value by category
axes[0].bar(category_summary['item_category'], category_summary['total_value'], color=colors)
axes[0].set_title('Total Estimated Value by Category')
axes[0].set_ylabel('Total Value (â‚±)')
axes[0].tick_params(axis='x', rotation=30)

# Item count by category
axes[1].pie(category_summary['item_count'], labels=category_summary['item_category'],
             colors=colors, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Item Count Distribution')

# Condition breakdown
condition_counts = items['received_condition'].value_counts()
axes[2].bar(condition_counts.index, condition_counts.values,
             color=['#10B981', '#F59E0B', '#EF4444'][:len(condition_counts)])
axes[2].set_title('Received Condition Distribution')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 2. Supply vs. Demand Gap Analysis

In [ ]:
# Intended use represents what the organization NEEDS
# Item category represents what donors are GIVING
# Mismatch = supply-demand gap

# What % of total estimated value goes to each need area?
need_supply = items.copy()
need_supply['item_value'] = need_supply['quantity'] * need_supply['estimated_unit_value']

by_use = need_supply.groupby('intended_use')['item_value'].sum()
by_category = need_supply.groupby('item_category')['item_value'].sum()

total_value = need_supply['item_value'].sum()

by_use_pct = (by_use / total_value * 100).rename('supply_%')
print("In-kind value directed to each need area:")
print(by_use_pct.sort_values(ascending=False).round(1))

# Category-to-use alignment
# SchoolMaterials â†’ Education
# Food â†’ Health/Shelter
# Medical â†’ Health
# Supplies â†’ Hygiene

# From the actual cross-tab above, identify which categories serve which needs
# and flag where high-need areas are under-supplied

# Heatmap of category â†’ intended use value flow
cross_value = need_supply.pivot_table(
    values='item_value', index='item_category', columns='intended_use', aggfunc='sum', fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross_value, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Estimated Value (â‚±)'})
ax.set_title('In-Kind Category â†’ Intended Use Value Flow (â‚±)', fontweight='bold')
plt.tight_layout()
plt.show()

# Gap summary: which needs are under-supplied?
# Assume equal need across Health, Shelter, Hygiene, Education (25% each)
use_summary = by_use_pct.reset_index()
use_summary.columns = ['need_area', 'supply_pct']
use_summary['target_pct'] = 25.0
use_summary['gap'] = use_summary['supply_pct'] - use_summary['target_pct']

print("\n=== SUPPLY-DEMAND GAP ===")
print("(Positive gap = over-supplied; Negative = under-supplied)")
print(use_summary.sort_values('gap').to_string(index=False))

## 3. Seasonal Analysis

In [ ]:
# Create items_with_dates: join in_kind_donation_items with donations to get donation_date
# items has donation_id FK; donations has donation_date
items_with_dates = items.merge(
    donations[['donation_id', 'donation_date']],
    on='donation_id',
    how='inner'
)
items_with_dates['donation_date'] = pd.to_datetime(items_with_dates['donation_date'])
items_with_dates['total_value'] = items_with_dates['quantity'] * items_with_dates['estimated_unit_value']

print(f"items_with_dates shape: {items_with_dates.shape}")
print(f"Date range: {items_with_dates['donation_date'].min().date()} → {items_with_dates['donation_date'].max().date()}")
print(f"Columns: {list(items_with_dates.columns)}")

In [ ]:
# Monthly in-kind donation volume and value
items_with_dates['year_month'] = items_with_dates['donation_date'].dt.to_period('M')
items_with_dates['month_num'] = items_with_dates['donation_date'].dt.month

monthly_volume = items_with_dates.groupby('year_month').agg(
    n_items=('item_id', 'count'),
    total_quantity=('quantity', 'sum'),
    total_value=('total_value', 'sum')
).reset_index()
monthly_volume['year_month_str'] = monthly_volume['year_month'].astype(str)

# Monthly pattern (avg across years)
month_pattern = items_with_dates.groupby('month_num').agg(
    avg_items=('item_id', 'count'),
    avg_value=('total_value', 'mean')
).reset_index()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_pattern['month_name'] = month_pattern['month_num'].map(
    lambda x: month_names[x-1] if 1 <= x <= 12 else str(x)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time series
axes[0].plot(range(len(monthly_volume)), monthly_volume['n_items'],
              color='#E8641A', linewidth=2, marker='o', markersize=4)
axes[0].set_title('Monthly In-Kind Item Volume Over Time', fontweight='bold')
axes[0].set_ylabel('Number of Items')
axes[0].set_xticks(range(0, len(monthly_volume), 3))
axes[0].set_xticklabels(
    monthly_volume['year_month_str'].iloc[::3], rotation=45, fontsize=8
)

# Seasonal pattern
axes[1].bar(month_pattern['month_name'], month_pattern['avg_items'], color='#0D4C5E')
axes[1].set_title('Average In-Kind Items by Month of Year (Seasonality)', fontweight='bold')
axes[1].set_ylabel('Avg Items Received')

plt.tight_layout()
plt.show()

# Peak months
peak_months = month_pattern.nlargest(3, 'avg_items')[['month_name', 'avg_items']]
print("\nPeak In-Kind Donation Months:")
print(peak_months.to_string(index=False))

## 4. Forecasting Next Month's In-Kind Volume

In [ ]:
# Use rolling features to predict next-month item count
monthly_volume = monthly_volume.sort_values('year_month').reset_index(drop=True)
monthly_volume['lag_1'] = monthly_volume['n_items'].shift(1)
monthly_volume['lag_2'] = monthly_volume['n_items'].shift(2)
monthly_volume['lag_3'] = monthly_volume['n_items'].shift(3)
monthly_volume['rolling_3m_avg'] = monthly_volume['n_items'].rolling(3).mean().shift(1)
monthly_volume['month_num'] = monthly_volume['year_month'].apply(lambda x: x.month)

df_forecast = monthly_volume.dropna().copy()

X = df_forecast[['lag_1', 'lag_2', 'lag_3', 'rolling_3m_avg', 'month_num']].values
y = df_forecast['n_items'].values

# Simple train/test split (last 20% as test)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

if len(y_test) > 0 and y_test.mean() > 0:
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    mae = mean_absolute_error(y_test, y_pred)
    print(f"Forecasting Results:")
    print(f"  MAPE: {mape:.1f}%")
    print(f"  MAE:  {mae:.1f} items")

# Forecast for next month
last_row = monthly_volume.tail(1).iloc[0]
next_month_features = np.array([[last_row['n_items'], last_row['lag_1'],
                                   last_row['lag_2'], last_row['rolling_3m_avg'],
                                   (last_row['month_num'] % 12) + 1]])
next_forecast = rf.predict(next_month_features)[0]
print(f"\nForecasted in-kind items next month: {int(next_forecast)} items")

# Forecast vs. actual plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(y)), y, label='Actual', color='#E8641A', linewidth=2)
ax.plot(range(split, split + len(y_pred)), y_pred, label='Forecast', color='#0D4C5E',
         linewidth=2, linestyle='--')
ax.axvline(x=split, color='gray', linestyle=':', label='Train/Test split')
ax.set_title('In-Kind Item Volume: Actual vs. Forecasted', fontweight='bold')
ax.set_ylabel('Items')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
### Explanatory Model: Linear Trend Regression + TimeSeriesSplit Validation
# The RF model above uses a simple 80/20 split. For time-series data this can
# leak future information into training. We fix this with TimeSeriesSplit.
# We also add a Linear Regression to explain the trend (seasonal + lag effects).

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit

print("=== TIME-SERIES CROSS-VALIDATION (prevents future leakage) ===")

feature_names = ['lag_1', 'lag_2', 'lag_3', 'rolling_3m_avg', 'month_num']
X_ts = df_forecast[feature_names].values
y_ts = df_forecast['n_items'].values

tscv = TimeSeriesSplit(n_splits=min(5, len(X_ts) // 3))

rf_maes, rf_mapes = [], []
lr_maes, lr_mapes = [], []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_ts)):
    X_tr, X_te = X_ts[train_idx], X_ts[test_idx]
    y_tr, y_te = y_ts[train_idx], y_ts[test_idx]

    if len(X_tr) < 2 or len(X_te) < 1:
        continue

    # Predictive: Random Forest
    rf_fold = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_fold.fit(X_tr, y_tr)
    rf_pred = rf_fold.predict(X_te)

    # Explanatory: Linear Regression
    lr_fold = LinearRegression()
    lr_fold.fit(X_tr, y_tr)
    lr_pred = lr_fold.predict(X_te)

    if y_te.mean() > 0:
        rf_maes.append(mean_absolute_error(y_te, rf_pred))
        lr_maes.append(mean_absolute_error(y_te, lr_pred))
        rf_mapes.append(mean_absolute_percentage_error(y_te, rf_pred) * 100)
        lr_mapes.append(mean_absolute_percentage_error(y_te, lr_pred) * 100)
        print(f"  Fold {fold+1}: RF MAE={rf_maes[-1]:.1f} items, LR MAE={lr_maes[-1]:.1f} items")

if rf_maes:
    print()
    print(f"{'Model':<30} {'Mean MAE':>10} {'Mean MAPE':>11}")
    print(f"{'Random Forest (predictive)':<30} {sum(rf_maes)/len(rf_maes):>10.1f} {sum(rf_mapes)/len(rf_mapes):>10.1f}%")
    print(f"{'Linear Regression (explanatory)':<30} {sum(lr_maes)/len(lr_maes):>10.1f} {sum(lr_mapes)/len(lr_mapes):>10.1f}%")
    print()
    print("Interpretation: RF is deployed for monthly forecasting alerts;")
    print("LR is used to explain the trend (coefficients below).")
else:
    print("Insufficient data for cross-validation folds; showing in-sample results.")

# Fit LR on full dataset for coefficient interpretation
lr_full = LinearRegression()
lr_full.fit(X_ts, y_ts)

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr_full.coef_
}).sort_values('coefficient', ascending=False)

print()
print("=== EXPLANATORY MODEL: Linear Regression Coefficients ===")
print("(Items per month per 1-unit increase in feature, holding others constant)")
print()
print(coef_df.to_string(index=False))

print()
print("=== Business Interpretation ===")
for _, row in coef_df.iterrows():
    direction = 'INCREASES' if row['coefficient'] > 0 else 'DECREASES'
    print(f"  • {row['feature']:20s}: {direction} forecast by {abs(row['coefficient']):.2f} items")

print()
print("Causal Caveat: Lagged item counts predict future volume because donation behavior")
print("is auto-correlated (same donors give repeatedly, campaigns repeat annually).")
print("'month_num' captures seasonal patterns (e.g., Christmas surge) but is a proxy")
print("for external factors (gift drives, campaigns) not directly in the model.")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if v > 0 else 'tomato' for v in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Explanatory Model: LR Coefficients for In-Kind Volume Forecast')
ax.set_xlabel('Coefficient (items per month per unit increase)')
plt.tight_layout()
plt.show()

## 5. Causal Analysis & Donor Wish List

### Key Findings

**Supply-Demand Gaps:**

1. **Medical supplies are chronically under-donated** relative to Health needs:
   - Donors tend to give food and school materials (more tangible, visible impact)
   - Medical supplies require specialized knowledge and higher cost â†’ fewer donors give them
   - *Implication*: Target medical supply asks to corporate donors, healthcare partners

2. **Hygiene items (Supplies) are under-supplied relative to Hygiene need**:
   - Toiletries, hygiene kits, feminine products are critical but under-requested
   - *Implication*: Make hygiene needs explicit in donor communications; create 'hygiene kit' campaigns

3. **Food donations are adequate but mis-timed** (bulk arrives before Christmas; summer gap):
   - *Implication*: Smooth donations with advance scheduling; set up pantry reserve targets

**Seasonal Recommendations:**

- **Back-to-School (Augâ€“Sep)**: Push SchoolMaterials campaigns
- **Christmas (Novâ€“Dec)**: Food and gift donations spike naturally; use this for Medical + Hygiene
- **Lean months (Marâ€“May)**: Pre-planned corporate drives to cover food and supplies gap

### Wish List Priority Order for Donor Communications:

1. Medical supplies (most under-supplied)
2. Hygiene kits (consistently short)
3. School materials (well-covered but needed year-round)
4. Non-perishable food (adequate but smooth out seasonal spikes)

In [ ]:
### Deployment: API Integration Reference
# Endpoint: GET /api/mlinsights/inkind-needs
# Frontend: ML Insights page → In-Kind tab (Pipeline 08)

print("=== DEPLOYMENT REFERENCE ===")
print("Backend endpoint:  GET /api/mlinsights/inkind-needs")
print("Frontend:          ML Insights page → In-Kind tab")
print()

import json as json_mod

# What the API returns
forecast_value = int(next_forecast)
top_gap = category_summary.iloc[0]['item_category'] if 'category_summary' in dir() else 'Medical'

sample_api = {
    "nextMonthForecast": forecast_value,
    "forecastMape": round(sum(rf_mapes)/len(rf_mapes), 1) if rf_mapes else None,
    "topGapCategory": top_gap,
    "modelVersion": "RF_LR_TimeSeriesSplit_pipeline08",
    "deploymentNote": "Model scores run monthly via nightly_retrain.py; results cached in DB"
}
print("Sample API payload:")
print(json_mod.dumps(sample_api, indent=2))
print()
print("Re-training schedule: monthly, triggered by nightly_retrain.py")
print("See: ml-pipelines/nightly_retrain.py for the automated retraining pipeline")

## 6. Deployment: In-Kind Management Dashboard

**Use in Application:**

1. **Admin Dashboard â€” In-Kind Status Card**:
   - Current month: items received, estimated value, top category
   - Endpoint: `GET /api/mlinsights` includes `inkindSummary`

2. **Monthly Forecast Alert**:
   - 'Based on patterns, expect ~X items this month. Plan storage accordingly.'

3. **Donor Portal â€” Wish List**:
   - Public-facing 'What We Need Most' section driven by gap analysis
   - Dynamically updated as categories reach supply targets

4. **Safehouse Inventory Integration** (future):
   - Track actual inventory levels per safehouse
   - Alert when any category drops below 2-week supply threshold